In [1]:

from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from finance_byu.summarize import summary
from finance_byu.regtables import Regtable
import statsmodels.formula.api as smf
from scipy import stats

In [ ]:
# HELPER FUNCTIONS
def CheckContinuity(df):
    full_range = pd.date_range(df['Date'].min(), df['Date'].max(),freq='MS')
    # Checking for gaps in dates that could throw off momentum
    return df['Date'].nunique() == len(full_range)


def LoadCleanFile(filename):
    df = pd.read_csv(filename,na_values=[-99.99,-999])

    df['Date'] = df['Date'].astype(str)

    # eliminate yearly data
    # df = df.loc[df.Date.apply(lambda x: len(x)) == 6]
    df['Date'] = df['Date'][df['Date'].apply(lambda x: len(x.split('.')[0])) == 6]
    # Get rid of nans
    df = df.query('Date == Date')

    # convert to datetime
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%m').values.astype("datetime64[M]")
    if len(df) != df.Date.nunique():
        df = df.drop_duplicates(subset='Date')

    
    if not CheckContinuity(df):
        print("WARNING: date range is not continuous.")

    return df.dropna()

# load in factor datasets
F3 = LoadCleanFile('FF3_factor_data.csv')
F5 = LoadCleanFile('FF5_factor_data.csv')

def CreateRollingIndex(df,window_size=6):
    start = df.Date.min()
    n_windows = df.Date.nunique()

    idx = pd.DatetimeIndex(
    pd.concat([
        pd.Series(pd.date_range(start + pd.DateOffset(months=i),
                                periods=window_size, freq="MS"))
        for i in range(n_windows)
    ]))

    cutoff = sum([i for i in range(1,window_size)])

    # make sure the last date where we can hold for 6 months is the last porfolio
    return idx[idx.isin(df.Date)][:-cutoff]


def RunRegressionAnalysis(results):
    # FF3 Regression
    # merge portfolio results to regress on
    reg_df = pd.merge(results, F3, on='Date')
    # get return over and above risk free rate
    reg_df['ret_oa'] = reg_df['ret'] - reg_df['RF']
    # fix problem with variable name (smf.old thought it was a minus sign)
    reg_df['Mkt'] = reg_df['Mkt-RF']

    # FF5 Regression
    reg1 = smf.ols('ret_oa ~ 1 + Mkt + SMB + HML',data=reg_df).fit()

    reg_df = pd.merge(results, F5, on='Date')
    reg_df['ret_oa'] = reg_df['ret'] - reg_df['RF']
    reg_df['Mkt'] = reg_df['Mkt-RF']

    reg2 = smf.ols('ret_oa ~ 1 + Mkt + SMB + HML + RMW + CMA',data=reg_df).fit()

    tbl = Regtable([reg1,reg2],stat='tstat',sig='coeff')
    return tbl.render()



# NOTES:
"""
According to Part B of the paper the momentum is calculated in the last 6 months and portolfios are created by investing in the top 3 momentum industries
and shorting the bottom 3 and holding for a period of 6 months. (this is for industries, it's top and bottom 30% for individual stocks)
"""

"\nAccording to Part B of the paper the momentum is calculated in the last 6 months and portolfios are created by investing in the top 3 momentum industries\nand shorting the bottom 3 and holding for a period of 6 months. (this is for industries, it's top and bottom 30% for individual stocks)\n"

# 17 Firms

In [3]:
df = LoadCleanFile('17_Industry_Portfolios.csv')
results17 = RunReturnAnalysis(df)

RunRegressionAnalysis(results17)

              ret
count  385.000000
mean     0.354409
std      3.055821
tstat    2.275662
pval     0.023416
min    -11.283889
25%     -1.341667
50%      0.404444
75%      2.218889
max      9.841667


,ret_oa,ret_oa
Intercept,-0.186,-0.242
,(-1.16),(-1.41)
Mkt,0.067,0.079
,(1.66),(1.91)
SMB,-0.029,0.004
,(-0.50),(0.08)
HML,-0.018,-0.107
,(-0.27),(-1.13)
RMW,,0.066
,,(0.50)


In [4]:
print(results17.index.min())
print(results17.index.max())


1963-07-01 00:00:00
1995-07-01 00:00:00


# 30 Firms

In [21]:
df = LoadCleanFile('30_Industry_Portfolios.csv')
results30 = RunReturnAnalysis(df)

RunRegressionAnalysis(results30)

(2280, 3)
              ret
count  385.000000
mean     0.620015
std      3.795508
tstat    3.205256
pval     0.001462
min    -11.691111
25%     -1.510556
50%      0.575556
75%      2.577222
max     15.606111


,ret_oa,ret_oa
Intercept,0.093,0.027
,(0.46),(0.13)
Mkt,0.038,0.059
,(0.75),(1.14)
SMB,-0.000,0.034
,(-0.00),(0.46)
HML,-0.035,-0.193
,(-0.43),(-1.65)
RMW,,0.050
,,(0.31)


In [19]:
def RunReturnAnalysis(df, start='1963-07', end='1995-07'):
    cols = df.columns[1:]
    df.iloc[:,1:] = df.iloc[:,1:].apply(pd.to_numeric, errors='coerce')
    mom_dict = {'Date':df['Date']}
    # creating momentum for each industry
    for name in cols:
        # I originally did this wrong, should be the cummulative product as outlined in the 
        # Jagadeesh and Titman paper. They normally shift 1 but we shifted 2 in class so I'm doing that.
        mom_dict[f"lagmom_{name}"] = df[name].transform(lambda x: 1 + x/100).shift(2).rolling(6,6).apply(lambda x: x.prod() - 1)
    # dropping where not enough values to get 6 full months of data
    mom_df = pd.DataFrame(mom_dict)
    mom_df = mom_df.dropna().reset_index(drop=True)

    df = df[(df['Date'] >= start) & (df['Date'] <= end)]
    # df = df[(df['Date'] >= '1995-08')]


    if len(df) > len(mom_df):
        df = df.loc[df.Date.isin(mom_df.Date)]
    elif len(df) < len(mom_df):
        mom_df = mom_df.loc[mom_df.Date.isin(df.Date)]


    if not CheckContinuity(df):
        print("WARNING: date range is not continuous in original dataframe.")
    if not CheckContinuity(mom_df):
        print("WARNING: date range is not continuous in momentum dataframe.")


    # get array of lagged industry momentum values to get portfolio choices
    # and then the array of returns to construct the portfolios
    array = mom_df.iloc[:,1:].values.astype(float)
    ret_array = df.iloc[:,1:].values.astype(float)


    hold_val = 6 # how long we'll hold each portfolio
    sorter = np.argsort(array,axis=1)
    # get choices per time period
    top3 = sorter[:-hold_val+1,-3:]
    top3 = np.array([np.repeat(a.reshape(1,-1), hold_val, axis=0) for a in top3]).reshape(-1,3)
    bot3 = sorter[:-hold_val+1,:3]
    bot3 = np.array([np.repeat(a.reshape(1,-1), hold_val, axis=0) for a in bot3]).reshape(-1,3)



    # get all values that will represent profits for 6-month held portfolios starting the day lagged momentum was decided
    broadcaster = np.array([i+j for i in range(len(ret_array)-hold_val+1) for j in range(hold_val)]).reshape(-1,1)
    print(ret_array[broadcaster,top3].shape)
    returns = ret_array[broadcaster,top3].mean(axis=1) - ret_array[broadcaster,bot3].mean(axis=1)

    # create rolling index to encompass multiple portfolios on multiple days.
    window_dates = CreateRollingIndex(df)
    port_rets = pd.DataFrame({'Date':window_dates,'part_ret':returns})

    results = (port_rets.groupby('Date')['part_ret'].mean().to_frame(name='ret'))

    print(summary(results))
    return results
    # return results.to_frame(name='ret')

# 48 Firms

In [20]:
df = LoadCleanFile('48_Industry_Portfolios_copy2.csv')
results48 = RunReturnAnalysis(df)

RunRegressionAnalysis(results48)

IndexError: shape mismatch: indexing arrays could not be broadcast together with shapes (1848,1) (1806,3) 

# Mean and Tstat Comparisons

In [ ]:
print(f"{"":15} {"Mean":>8} {"Tstat":>8}")
print(f"{"17 Industries":15} {results17['ret'].mean():>8.3f} {results17['ret'].mean() / (results17['ret'].std() / np.sqrt(len(results17))):>8.3}")
print(f"{"30 Industries":15} {results30['ret'].mean():>8.3f} {results30['ret'].mean() / (results30['ret'].std() / np.sqrt(len(results30))):>8.3}")
print(f"{"48 Industries":15} {results48['ret'].mean():>8.3f} {results48['ret'].mean() / (results48['ret'].std() / np.sqrt(len(results48))):>8.3}")

                    Mean    Tstat
17 Industries      0.428     1.81
30 Industries      0.685      2.3
48 Industries      0.695     2.04
